# 20 — Readout Leakage Benchmarking

Legacy experiment 26.

Quantify measurement-induced leakage from the readout resonator to the qubit
and the storage cavity, and benchmark the readout QND-ness.

This is a characterization-only notebook — no calibrations are applied.

## 1. Shared Session Bootstrap

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from qualang_tools.units import unit

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "qubox").exists():
    REPO_ROOT = Path(r"E:\qubox")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from qubox.notebook import (
    load_stage_checkpoint,
    open_notebook_stage,
    save_stage_checkpoint,
)

REGISTRY_BASE = Path(r"E:\qubox")
SAMPLE_ID = "post_cavity_sample_A"
COOLDOWN_ID = "cd_2025_02_22"
QOP_IP = "10.157.36.68"
CLUSTER_NAME = "Cluster_2"

stage = open_notebook_stage(
    stage_name="20_readout_leakage_benchmarking",
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    qop_ip=QOP_IP,
    cluster_name=CLUSTER_NAME,
    force_reopen=True,
    close_existing=True,
)
session = stage.session
attr = stage.attr
SESSION_BOOTSTRAP_PATH = stage.bootstrap_path
u = unit(coerce_to_integer=True)

readout_checkpoint = load_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="16_readout_calibration",
)

print(f"Repo root on sys.path: {REPO_ROOT}")
print(f"Shared session bootstrap: {SESSION_BOOTSTRAP_PATH}")
print(f"Stage checkpoint path: {stage.checkpoint_path}")
print(f"QM endpoint: {QOP_IP} ({CLUSTER_NAME})")
if readout_checkpoint is not None:
    print(f"Prior stage 16 status: {readout_checkpoint['status']}")

## 2. Leakage Defaults

In [ ]:
LEAKAGE_N_AVG = 50000
LEAKAGE_N_READOUTS = [1, 5, 10, 20, 50]

print("Readout leakage benchmarking settings:")
print(f"  n_avg: {LEAKAGE_N_AVG}")
print(f"  n_readouts sweep: {LEAKAGE_N_READOUTS}")

## 3. Readout-Induced Leakage — Exp 26

Repeat readout multiple times and measure qubit state change after each.

In [ ]:
leakage_results = {}

for n_readouts in LEAKAGE_N_READOUTS:
    result = session.readout_leakage_measurement(
        n_readouts=n_readouts,
        n_avg=LEAKAGE_N_AVG,
    )
    leakage_results[n_readouts] = result
    print(f"  n_readouts={n_readouts}: done")

print("Readout leakage sweep complete.")

## 4. Leakage Summary Plot

In [ ]:
# Plot leakage probability vs. number of readouts
n_ro_list = sorted(leakage_results.keys())

fig, ax = plt.subplots(figsize=(8, 5))
# Placeholder: fill with actual leakage metrics from results
ax.set_xlabel("Number of consecutive readouts")
ax.set_ylabel("Leakage probability")
ax.set_title("Readout-Induced Leakage")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Leakage summary plotted (update with actual metrics from results).")

## 5. Save Checkpoint

In [ ]:
stage_checkpoint_path = save_stage_checkpoint(
    registry_base=REGISTRY_BASE,
    sample_id=SAMPLE_ID,
    cooldown_id=COOLDOWN_ID,
    stage_name="20_readout_leakage_benchmarking",
    status="characterized",
    summary="Readout-induced leakage benchmarked across multiple consecutive readout counts.",
    consumed_inputs={
        "leakage_n_avg": LEAKAGE_N_AVG,
        "leakage_n_readouts": LEAKAGE_N_READOUTS,
    },
    persisted_outputs={},
    advisory_outputs={},
    next_stage="21_storage_cavity_characterization",
    notes=[
        "Characterization-only — no calibrations applied.",
        "Results inform whether active reset is needed after measurement.",
    ],
    metrics={},
)

print(f"Stage checkpoint saved to: {stage_checkpoint_path}")